# Notebook-Zelle: DB laden + Filter

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

MARIADB_URL = os.getenv("MARIADB_URL")
assert MARIADB_URL, "MARIADB_URL env fehlt im heatmap pod!"

engine = create_engine(MARIADB_URL, pool_pre_ping=True)

TABLE = "forecast_predictions"   # ggf. anpassen!
TS_COL = "timestamp_hour_local"  # euer Timestamp-String

start_date = "2026-01-07"
end_date   = "2026-01-08"

# Wir laden bewusst etwas breiter und filtern in pandas sauber auf datetime
sql = f"""
SELECT
  row_i AS row,
  col_i AS col,
  {TS_COL} AS time,
  pred_event_count_int
FROM {TABLE}
"""

df = pd.read_sql(sql, engine)
print("Rows from DB:", len(df))
df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)

start_ts = pd.to_datetime(start_date, utc=True)
end_ts   = pd.to_datetime(end_date, utc=True) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

forecast_df_filt = df[(df["time"] >= start_ts) & (df["time"] <= end_ts)].copy()

print("Gefilterte Zeilen:", len(forecast_df_filt))
print("Zeitraum:", forecast_df_filt["time"].min(), "→", forecast_df_filt["time"].max())
forecast_df_filt.head()


# Notebook-Zelle: Grid bauen + Deutschland + Plot

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

# 1) Raster aus CSV
GRID_PATH = "/app/notebooks/de_grid_sym_400.csv"
df_grid = pd.read_csv(GRID_PATH)
df_grid["lat"] = df_grid["lat"].astype(float)
df_grid["lon"] = df_grid["lon"].astype(float)

lats = sorted(df_grid["lat"].unique())
lons = sorted(df_grid["lon"].unique())

cells = []
for j in range(len(lats) - 1):
    for i in range(len(lons) - 1):
        lat_min, lat_max = lats[j], lats[j + 1]
        lon_min, lon_max = lons[i], lons[i + 1]
        geom = box(lon_min, lat_min, lon_max, lat_max)
        cells.append({"row": j, "col": i, "geometry": geom})

grid_gdf = gpd.GeoDataFrame(cells, crs="EPSG:4326")

# 2) Deutschland-Grenze (lokal via naturalearth_lowres)
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
de = world[world["name"] == "Germany"].to_crs("EPSG:4326")

# 3) Aggregation
agg = (
    forecast_df_filt
    .groupby(["row", "col"], as_index=False)["pred_event_count_int"]
    .sum()
    .rename(columns={"pred_event_count_int": "pred_event_sum"})
)

# 4) Merge ins Raster
grid_with_vals = grid_gdf.merge(agg, on=["row", "col"], how="left")
grid_with_vals["pred_event_sum"] = grid_with_vals["pred_event_sum"].fillna(0)

# optional: Clip auf Deutschland
grid_with_vals = gpd.overlay(grid_with_vals, de[["geometry"]], how="intersection")

# 5) Plot
de_proj = de.to_crs("EPSG:3857")
grid_proj = grid_with_vals.to_crs("EPSG:3857")

fig, ax = plt.subplots(figsize=(8, 12))
de_proj.boundary.plot(ax=ax, linewidth=1.0)

grid_proj.plot(
    ax=ax,
    column="pred_event_sum",
    cmap="Reds",
    linewidth=0,
    legend=True,
)

ax.set_aspect("equal")
ax.set_title(f"Vorhergesagte Stauwarnungen (Summe) {start_date} bis {end_date}")
ax.axis("off")
plt.tight_layout()
plt.show()
